In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.metrics import f1_score
from sklearn.ensemble import HistGradientBoostingClassifier
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline
import joblib
from time import time

# Locate dataset (robust search through parents)
project_root = Path.cwd()
data_path = None
for parent in [project_root] + list(project_root.parents):
    candidate = parent / 'Data' / 'Numerical' / 'Data' / '65_Nutrients_Data.csv'
    if candidate.exists():
        data_path = candidate
        project_root = parent
        break

if data_path is None:
    raise FileNotFoundError('Could not find 65_Nutrients_Data.csv under any parent of '+str(Path.cwd()))

print('Using data file:', data_path)
df = pd.read_csv(data_path)
print('df shape:', df.shape)
if 'novaclass' not in df.columns:
    raise KeyError('novaclass column missing in dataset')
X = df.drop(columns=['novaclass'])
y = df['novaclass'].astype(int)
min_label = int(y.min())
if min_label != 0:
    y = y - min_label
    print('Shifted labels by', min_label)

# train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print('train/test sizes', X_train.shape, X_test.shape)

# cross-validator and pipeline
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
sm = SMOTE(random_state=42)
hgb = HistGradientBoostingClassifier(random_state=42)
pipe_hgb = Pipeline([('smote', sm), ('hgb', hgb)])

param_dist_hgb = {
    'hgb__max_iter': [100, 200, 400],
    'hgb__max_depth': [None, 8, 12],
    'hgb__learning_rate': [0.03, 0.1],
}
rs_hgb = RandomizedSearchCV(pipe_hgb, param_distributions=param_dist_hgb, n_iter=6, scoring='f1_macro', cv=cv, n_jobs=-1, random_state=42, verbose=1, return_train_score=True)
print('Starting HistGradientBoosting RandomizedSearchCV (6 trials)')
t0 = time()
rs_hgb.fit(X_train, y_train)
t1 = time()
print('Done. Best CV f1_macro (HGB):', rs_hgb.best_score_)
print('Best params (HGB):', rs_hgb.best_params_)
best_hgb = rs_hgb.best_estimator_
y_pred_hgb = best_hgb.predict(X_test)
print('Test f1_macro (HGB):', f1_score(y_test, y_pred_hgb, average='macro'))
print('Search time (s):', round(t1-t0,2))
out_dir_hgb = project_root / 'models' / 'numerical_65_hgb' / 'smote'
out_dir_hgb.mkdir(parents=True, exist_ok=True)
joblib.dump(best_hgb, out_dir_hgb / 'hgb_65_numerical_smote.joblib')
print('Saved HGB model to', out_dir_hgb)

Using data file: c:\Users\Divyeh\OneDrive\Desktop\ML_IP\NOVA_Food_Processing\Data\Numerical\Data\65_Nutrients_Data.csv
df shape: (2970, 66)
Shifted labels by 1
train/test sizes (2376, 65) (594, 65)
Starting HistGradientBoosting RandomizedSearchCV (6 trials)
Fitting 5 folds for each of 6 candidates, totalling 30 fits
Done. Best CV f1_macro (HGB): 0.8396810176205409
Best params (HGB): {'hgb__max_iter': 200, 'hgb__max_depth': 8, 'hgb__learning_rate': 0.1}
Test f1_macro (HGB): 0.8620893350478517
Search time (s): 160.75
Saved HGB model to c:\Users\Divyeh\OneDrive\Desktop\ML_IP\NOVA_Food_Processing\models\numerical_65_hgb\smote
